# T3-DiffWeather Colab Test (ECCV 2024)
**Teaching Tailored to Talent: Adverse Weather Restoration via Prompt Pool and Depth-Anything Constraint**

이 노트북은 `T3-DiffWeather`의 데모 테스트를 위한 노트북입니다.
- Google Drive Mount를 지원합니다.
- 논문에서 제공하는 공식 Pre-trained 가중치를 사용합니다.
- 벤치마크 테스트 파일과 코드를 제외하고 오직 커스텀 이미지 결과 도출을 목적으로 작성되었습니다.

In [ ]:
# ===== 1. Colab GPU 확인 =====
!python -V
!nvidia-smi

In [ ]:
# ===== 2. Google Drive 마운트 =====
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== 3. 환경 설정 및 리포지토리 클론 =====
%cd /content
!git clone https://github.com/Ephemeral182/ECCV24_T3-DiffWeather.git
%cd /content/ECCV24_T3-DiffWeather

# 요구사항 설치 (공식 requirements.txt + colab용 유틸리티 다운로더 등)
!pip install -q numpy opencv-python scikit-image tqdm matplotlib tensorboardX pytorch_lightning==2.0.4 timm einops pillow diffusers gdown transformers

In [ ]:
# ===== 4. Pretained 가중치 다운로드 =====
import os
import gdown

os.makedirs("/content/ECCV24_T3-DiffWeather/pretrained", exist_ok=True)

# T3_DiffWeather_demo.ckpt 논문 제공 공식 가중치
file_id = "1T1SGbHxPwdlqk5rs0-4lD5PVX4SCkpry"
url = f"https://drive.google.com/uc?id={file_id}"
download_path = "/content/ECCV24_T3-DiffWeather/pretrained/T3_DiffWeather_demo.ckpt"

if not os.path.exists(download_path):
    print("가중치 다운로드를 시작합니다...")
    gdown.download(url, download_path, quiet=False)
    print("다운로드 완료:", download_path)
else:
    print("이미 가중치가 존재합니다:", download_path)

In [ ]:
# ===== 5. 테스트용 입력 경로 탐색 및 Config 수정 =====
import yaml
import os
import glob

def find_render_dir(base_dir):
    """
    구조화된 경로 내에서 렌더링 결과 폴더를 자동으로 탐색합니다.
    패턴: base_dir/**/test/ours_*/renders
    """
    search_pattern = os.path.join(base_dir, "**", "test", "ours_*", "renders")
    matches = glob.glob(search_pattern, recursive=True)

    if matches:
        return sorted(matches)[-1]

    for root, dirs, files in os.walk(base_dir):
        if 'crop' in root:
            continue
        if any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in files):
            return root

    return base_dir

# TODO: 악천후 이미지가 저장된 구글 드라이브 기본 경로(BASE_DIR)를 지정하세요.
# 예시: BASE_DIR = "/content/drive/MyDrive/Research/26_Capstone2/LongSplat/snow"
BASE_DIR = "/content/test_images"  # 임시 기본값
os.makedirs(BASE_DIR, exist_ok=True)

# 자동으로 렌더링 이미지 디렉토리 탐색
test_image_dir = find_render_dir(BASE_DIR)
print(f"[*] 발견된 이미지 디렉토리: {test_image_dir}")

# 데모 테스트를 위한 Config 파일(`allweather_demo.yml`) 내용 수정
config_path = "/content/ECCV24_T3-DiffWeather/configs/allweather_demo.yml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# 논문상 기본 eval 코드는 모델에 val_data_dir에 있는 모든 이미지를 넣습니다.
config['data']['val_data_dir'] = test_image_dir

with open(config_path, 'w') as f:
    yaml.dump(config, f)

print(f"[*] Config 파일 수정 완료. 모델이 다음 디렉토리의 이미지를 읽어옵니다: {test_image_dir}")
print("💡 드라이브 베이스 경로를 따로 지정하지 않았다면, 좌측 탭을 통해 `/content/test_images` 에 이미지를 업로드하세요.")

In [ ]:
# ===== 6. T3-DiffWeather 모델 실행 (이미지 복원) =====
%cd /content/ECCV24_T3-DiffWeather

# Depth-Anything Vits14 가 백그라운드 환경으로 다운로드 되며, inference가 진행됩니다.
!python test_diffuser_demo.py --config configs/allweather_demo.yml --model_path pretrained/T3_DiffWeather_demo.ckpt

In [ ]:
# ===== 7. 결과 확인 및 시각화 =====
import matplotlib.pyplot as plt
from PIL import Image
import glob
import os

results_dir = "/content/ECCV24_T3-DiffWeather/save_images_test_Demo"
result_images = sorted(glob.glob(os.path.join(results_dir, "sample_*.png")))

if not result_images:
    print("테스트 된 결과 이미지가 없습니다. 입력 이미지를 제대로 넣었는지 확인하세요.")
else:
    print(f"총 {len(result_images)}개의 이미지가 성공적으로 복원되었습니다.\n")
    for img_path in result_images:
        img = Image.open(img_path)
        # sample_ 접두사가 붙어 나옵니다.
        id_name = os.path.basename(img_path).replace("sample_", "").replace(".png", "")

        # 입력 이미지와 비교 (동일한 이름 기반 탐색)
        original_img_cands = glob.glob(os.path.join(test_image_dir, f"{id_name}.*"))

        if len(original_img_cands) > 0:
            orig_img = Image.open(original_img_cands[0])
            fig, ax = plt.subplots(1, 2, figsize=(15, 7))
            ax[0].imshow(orig_img)
            ax[0].set_title(f"Input: {os.path.basename(original_img_cands[0])}")
            ax[0].axis("off")

            ax[1].imshow(img)
            ax[1].set_title("Output (T3-DiffWeather)")
            ax[1].axis("off")
            plt.show()
        else:
            # 원본 이미지를 확장자 차이 등으로 못 찾을 경우 결과만 시각화
            plt.figure(figsize=(7, 7))
            plt.imshow(img)
            plt.title(f"Output (T3-DiffWeather): {id_name}")
            plt.axis("off")
            plt.show()